# singular-matrix-mask-trick — ex2: block-diagonal partial-singular solve — detect per-block

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `singular-matrix-mask-trick`. Running the final beacon cell reports progress against the `Numpy: Singular matrix mask trick` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Singular matrix mask trick` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`singular-matrix-mask-trick`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "singular-matrix-mask-trick"
DD_SUBTOPIC = "Numpy: Singular matrix mask trick"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## singular-matrix mask trick — quick refresher

When a batched `linalg.solve` would otherwise crash on singular slices, the standard rescue is:

1. Detect singular slices: `is_sing = la.det(A).abs() < eps`.
2. Overwrite those slices with the identity (any safe invertible matrix would do): `A_safe[is_sing] = t.eye(N)`.
3. Solve everywhere; return `(x, is_valid)` so the caller can mask the bogus rows downstream.

**This drill (ex2) vs ex1.** ex1 ran the trick over a flat batch of `(B, N, N)` matrices (whole-slice singular). ex2 extends it to a **block-diagonal** structure: the matrix decomposes into two `(N/2, N/2)` blocks, and singularity is detected **per block** (some blocks invertible, others not). The trick generalises one level — same det/identity-replace pattern, applied to a block-batched view.

### Exercise 2 — block-diagonal partial-singular solve — detect per-block

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the singular-matrix-mask trick at block granularity: given a block-diagonal batch `(B, 2*N, 2*N)`, split into a `(B, 2, N, N)` block view, detect singular blocks via `la.det`, identity-replace them, and solve.
> Keywords: block-diagonal, per-block-singular, linalg-solve, partial-rescue
> ```

**KCs targeted:** `singular-detect-via-det`, `singular-overwrite-identity`

Implement `ex2_block_diag_solve(A, b, N, eps=1e-8)`.

Each `A[i]` is a `(2*N, 2*N)` BLOCK-DIAGONAL matrix with two `(N, N)` blocks: `A[i] = block_diag(A[i, :N, :N], A[i, N:, N:])`. Some blocks may be singular but not necessarily both.

**Steps.**
1. Extract the two diagonal blocks into a tensor of shape `(B, 2, N, N)` (block index 0 is top-left, block index 1 is bottom-right).
2. Extract the matching segments of `b` into `(B, 2, N)`.
3. Compute `dets = la.det(blocks)` — shape `(B, 2)`.
4. Build `is_valid = dets.abs() >= eps` — shape `(B, 2)`.
5. Identity-replace invalid blocks: `blocks_safe[~is_valid] = eye(N)` (use fancy indexing).
6. `x = la.solve(blocks_safe, b_blocks)` — shape `(B, 2, N)`.
7. Return `(x, is_valid)` — the per-block solution and per-block validity mask. The caller is expected to zero or ignore the `~is_valid` slices.

Inputs:
- `A`: `(B, 2*N, 2*N)` block-diagonal float tensor.
- `b`: `(B, 2*N)` float tensor.
- `N`: block size (int).
- `eps`: singularity threshold.

Output: `(x, is_valid)` — `x` is `(B, 2, N)`, `is_valid` is `(B, 2)` bool.

In [ ]:
def ex2_block_diag_solve(A: Tensor, b: Tensor, N: int, eps: float = 1e-8):
    import torch.linalg as la
    # (B, 2*N, 2*N) -> stack the two diagonal blocks -> (B, 2, N, N)
    blocks = t.stack([A[:, :N, :N], A[:, N:, N:]], dim=1)
    b_blocks = t.stack([b[:, :N], b[:, N:]], dim=1)
    # Per-block singular detection.
    dets = la.det(blocks)             # (B, 2)
    is_valid = dets.abs() >= eps      # (B, 2)
    # Identity-replace the invalid blocks.
    blocks_safe = blocks.clone()
    eye = t.eye(N, dtype=blocks.dtype)
    blocks_safe[~is_valid] = eye
    # Solve everywhere; caller masks bogus slices via is_valid.
    x = la.solve(blocks_safe, b_blocks)
    return x, is_valid


<details><summary>Solution</summary>

```python
def ex2_block_diag_solve(A: Tensor, b: Tensor, N: int, eps: float = 1e-8):
    import torch.linalg as la
    # (B, 2*N, 2*N) -> stack the two diagonal blocks -> (B, 2, N, N)
    blocks = t.stack([A[:, :N, :N], A[:, N:, N:]], dim=1)
    b_blocks = t.stack([b[:, :N], b[:, N:]], dim=1)
    # Per-block singular detection.
    dets = la.det(blocks)             # (B, 2)
    is_valid = dets.abs() >= eps      # (B, 2)
    # Identity-replace the invalid blocks.
    blocks_safe = blocks.clone()
    eye = t.eye(N, dtype=blocks.dtype)
    blocks_safe[~is_valid] = eye
    # Solve everywhere; caller masks bogus slices via is_valid.
    x = la.solve(blocks_safe, b_blocks)
    return x, is_valid
```

**Why per-block-level detection is the right granularity.** A block-diagonal matrix's full determinant is the product of its block determinants — so if EITHER block is singular the full `la.det` flags the whole slice as singular and ex1's whole-slice mask would throw away the good block too. Detecting per-block recovers more information without losing crash safety.

**Why `blocks_safe = blocks.clone()`.** Fancy-indexing assignment `blocks_safe[~is_valid] = eye` writes into the source storage, so cloning before mutating preserves the caller's `A`.

**Difference from ex1.** ex1 ran the det-singular-detect / identity-replace / solve trick over a flat batch of `(B, N, N)` slices — one validity flag per matrix. ex2 does the SAME trick but after first decomposing each matrix into its block-diagonal pieces, producing one validity flag per BLOCK. The structural pattern (detect → identity-replace → solve → return mask) is preserved; what changes is the unit of detection.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()